# OCR Technologies

**Module:** 15 — VLMs & Multimodal

Classic OCR stacks versus OCR-free VLMs — quality levers, errors, and hybrids.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Describe classic OCR: detect → recognize → postprocess
- Contrast OCR-free VLM reading with engine OCR
- Apply DPI/deskew/contrast/dictionary quality tips
- Score OCR with CER/WER and reconcile money fields


## Classic OCR Stack

### Definition
OCR converts images of text into machine text, often with boxes and confidences.

### Why it matters
Archives, compliance, and redaction need predictable geometry.

### How it works
Preprocess → detect lines/words → recognize → dictionary/LM postprocess → export (TXT/hOCR).

### Intuition
OCR is a specialized reader; VLMs are generalists that also read.

### Pitfalls
- Low-DPI phone photos
- Ignoring language hints
- Print engines on handwriting

### When to use
Bulk digitization, searchable PDFs, word-box redaction.


### Classic vs OCR-free VLM

| Dimension | Classic OCR | VLM reading |
|-----------|-------------|-------------|
| Output | Text+boxes | Text/JSON |
| Layout export | Strong | Prompt-dependent |
| Novel layouts | Rules-heavy | Flexible |
| Scale cost | Usually lower | Token-heavy |
| Errors | Wrong chars | Fluent invention |

```mermaid
flowchart LR
  P[Preprocess] --> D[Detect] --> R[Recognize] --> PP[Postprocess]
  I[Page image] --> M[VLM] --> J[Text/JSON]
```


In [ ]:
# Demo 1: dictionary correction
from difflib import get_close_matches
VOCAB = {"invoice","total","amount","date","vendor","subtotal","tax","due"}
def correct(tok):
    t = tok.lower()
    if t in VOCAB: return t
    m = get_close_matches(t, VOCAB, n=1, cutoff=0.7)
    return m[0] if m else tok
print([correct(t) for t in "Inv0ice Tota1 Am0unt".split()])


In [ ]:
# Demo 2: Levenshtein + CER
def levenshtein(a,b):
    dp = list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        prev, dp[0] = dp[0], i
        for j,cb in enumerate(b,1):
            cur = dp[j]
            dp[j] = prev if ca==cb else 1+min(prev, dp[j], dp[j-1])
            prev = cur
    return dp[-1]

def cer(ref, hyp):
    return 0.0 if not ref and not hyp else (1.0 if not ref else levenshtein(ref,hyp)/len(ref))

ref = "TOTAL DUE 19.99 USD"
for hyp in [ref, "TOTAL DUE 19.99 USO", "T0TAL DUE 1999 USD"]:
    print(repr(hyp), "CER", round(cer(ref,hyp),3))


In [ ]:
# Demo 3: WER
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    # reuse char levenshtein on word sequences via join markers
    return cer("\0".join(r), "\0".join(h)) if r else 0.0
print("WER", round(wer("total due nineteen", "total due twenty"), 3))


## OCR-Free VLMs

### Definition
VLMs read via visual tokens without a separate recognition head — often inside VQA/JSON extract.

### Why it matters
Strong on messy photos/UIs; weaker alone when you need character-level audit trails.

### How it works
Prompt for exact transcription; mark uncertain spans; cross-check critical fields with OCR.

### Intuition
Reading and interpreting are different jobs — separate them when stakes are high.

### Pitfalls
- Fluent wrong totals
- Silent language mismatch
- No boxes for redaction

### When to use
Flexible DocQA and mobile capture; hybridize for finance/KYC.


In [ ]:
# Demo 4: hybrid money reconcile
import re
def parse_money(s):
    m = re.search(r"(-?\d+[.,]\d{2})", s.replace(",",""))
    return float(m.group(1)) if m else None

def reconcile(ocr_text, vlm_value, tol=0.01):
    o, v = parse_money(ocr_text), parse_money(str(vlm_value))
    if o is None and v is None: return {"status":"HITL","reason":"unparsed"}
    if o is None: return {"status":"vlm","value":v}
    if v is None: return {"status":"ocr","value":o}
    if abs(o-v)<=tol: return {"status":"agree","value":o}
    return {"status":"HITL","ocr":o,"vlm":v}
print(reconcile("TOTAL 19.99","19.99"), reconcile("TOTAL 19.99","29.99"))


### Quality tips

| Lever | Guidance |
|-------|----------|
| DPI | ≥300 for small print |
| Orientation | Deskew / EXIF |
| Contrast | Adaptive threshold on fades |
| Crops | Field-level ROI OCR |
| Language | Set expected packs |
| Dictionaries | SKUs / meds / names |
| Consistency | Checksums, VAT math |
| Logging | Engine+version+preprocess hash |


In [ ]:
# Demo 5: preprocess plan
def preprocess_plan(dpi, mean_luma, skew_deg):
    steps = []
    if dpi < 200: steps.append("upsample_to_300")
    if skew_deg > 1.5: steps.append("deskew")
    if mean_luma > 0.85: steps.append("adaptive_threshold")
    if mean_luma < 0.25: steps.append("brighten")
    return steps + ["run_ocr"]
print(preprocess_plan(150,0.9,3.0))
print(preprocess_plan(300,0.5,0.2))


In [ ]:
# Demo 6: VLM transcription prompt + mock response
import json
req = {"model":"gpt-4o","messages":[{"role":"user","content":[
    {"type":"text","text":"Transcribe ALL readable text verbatim. Return JSON {lines:[], uncertain:[]}"},
    {"type":"image_url","image_url":{"url":"https://example.com/sign.jpg"}},
]}]}
mock = {"lines":["STOP","EXCEPT RIGHT TURN"], "uncertain":["EXCEPT"]}
print(json.dumps(req)[:160], "...\n", mock)


In [ ]:
# Demo 7: PII redaction using OCR boxes
import re
words = [
    {"text":"John","x":0.1,"y":0.1,"w":0.1,"h":0.05},
    {"text":"SSN","x":0.1,"y":0.2,"w":0.08,"h":0.05},
    {"text":"123-45-6789","x":0.2,"y":0.2,"w":0.3,"h":0.05},
]
def redact_targets(words):
    return [w for w in words if re.search(r"\d{3}-\d{2}-\d{4}", w["text"]) or w["text"].lower() in {"ssn"}]
print(redact_targets(words))


### Checklist — OCR quality gate

- [ ] DPI/orientation checked
- [ ] CER measured on sample
- [ ] Critical fields hybrid-checked
- [ ] Language set
- [ ] Engine version pinned


### Try it yourself — OCR quality

1. Parse `EUR 19,99` style amounts.
2. Ask VLM for `{text, uncertain_spans[]}`.
3. Paint/redact boxes matching PII regex (extend Demo 7).

**Stretch:** Compare CER before/after deskew on a synthetic skewed line.


### Try it yourself — Hybrid design

1. Define HITL policy table for agree/disagree/unparsed.
2. Log both OCR and VLM raw strings for audit.


## Knowledge Check

**Q1.** Why can VLM OCR be dangerous for money?

<details><summary>Answer</summary>

It may invent fluent, plausible digit strings without character-level evidence.

</details>

**Q2.** What does CER normalize by?

<details><summary>Answer</summary>

Number of characters in the reference string.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `OCR` | Optical character recognition |
| `CER` | Character error rate |
| `WER` | Word error rate |
| `hOCR` | HTML OCR annotation format |
| `deskew` | Correct scan tilt |
| `ROI` | Region of interest |


## Key Takeaways

- Classic OCR: auditable text+boxes at scale
- VLMs read flexibly but can hallucinate digits
- Hybrids + math/dictionary checks catch costly errors
- Preprocess often beats model swaps


## Production Incident Patterns — OCR

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "OCR",
}))


## Mini Case Study — OCR

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("OCR", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — OCR ops

1. Write a one-page runbook section for on-call when OCR critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
